# Training the latest models: prn_1 / proj_1

Standalone notebook: preparation and labeling code is embedded; the read_prediction package is not required. Full training requires numpy, lightgbm and msr-cambridge1.tar in DATASET.

- Train: days 1–3 `[0, 72h)`; validation: day 4 `[72h, 96h)`. Labels in each window cannot use events after its end.
- same_start requires matching starting Offsets; range requires overlapping byte ranges.
- Train log-L2 on log1p(seconds) and raw-L1 on seconds for both rules. All seven causal features are transformed with log1p.
- proj_1 candidates: log-L2/raw-L1/per-address median. prn_1 additionally includes three mixtures: 75/25, 50/50, 25/75, after converting predictions to seconds.
- Select minimum validation MAE, breaking ties by method name. Pass fraction does not constrain selection. Models are not refitted on validation.

The published run selected 25% log-L2 + 75% raw-L1 for prn_1 and the per-address median for proj_1. This notebook repeats selection; future days 5–7 are not loaded here. The development-day 5–7 conclusions about MAE < mean overwrite and failure to reach ≥90% are in results.ipynb.

In [ ]:
from array import array
from pathlib import Path
from tempfile import TemporaryDirectory
import gzip
import hashlib
import importlib.util
import json
import tarfile

import lightgbm as lgb
import numpy as np

notebook_directory = Path('submission') if Path('submission/training.ipynb').is_file() else Path('.')
notebook_directory = notebook_directory.resolve()
DATASET = notebook_directory.parent / 'dataset'
OUTPUT = notebook_directory / 'trained' / 'long-history-retrain'
WORKLOADS = ('prn_1', 'proj_1')
SMOKE_TEST = False  # True: artificial trace only, not measured results.
print('DATASET:', DATASET, '\nOUTPUT:', OUTPUT)

## Seven features available at Write time

Offset (bytes), size (bytes), previous Read/Write counts, seconds since the last Read/Write, and the last completed Read-to-Read interval. History is keyed by the same starting Offset and is available before the current Write, including under the range rule. Missing history is NaN. ResponseTime and future events are not features. Writes without a Read before the end of their own window are excluded from regression; coverage is reported separately.

In [ ]:
TICKS_PER_SECOND = TICKS = 10_000_000
BLOCK = 512
_INDEX_BITS = 31
_INDEX_MASK = 2147483647
_INDEX_MOD = 2147483648
FEATURE_NAMES = [
    'log_offset_bytes',
    'log_size_bytes',
    'log_prior_read_count',
    'log_prior_write_count',
    'log_seconds_since_read',
    'log_seconds_since_write',
    'log_last_read_read_interval_seconds',
]
RAW_FEATURES = [
    'offset_bytes',
    'size_bytes',
    'prior_read_count',
    'prior_write_count',
    'seconds_since_read',
    'seconds_since_write',
    'last_read_read_interval_seconds',
]
EVENT_DTYPE = np.dtype([('timestamp', '<i8'), ('offset', '<i8'), ('size', '<i8'), ('is_read', '<i8')])
TRACE_SOURCES = {
    'prn_1': ('msr-cambridge1.tar', 'MSR-Cambridge/prn_1.csv.gz'),
    'proj_1': ('msr-cambridge1.tar', 'MSR-Cambridge/proj_1.csv.gz'),
}
PARAMETERS = {
    'n_estimators': 150,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 50,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': 2,
    'deterministic': True,
    'force_col_wise': True,
    'verbosity': -1,
}
MIX_WEIGHTS = {
    'mix_log75': 0.75,
    'mix_log50': 0.5,
    'mix_log25': 0.25,
}
INFERENCE_SOURCE = '"""Standalone inference for validation-selected mixtures, regressors and medians."""\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport lightgbm as lgb\nimport numpy as np\n\n\nclass SelectedPredictor:\n    """Accept seven raw causal features; return nonnegative seconds to Read."""\n\n    def __init__(self, directory=None):\n        self.directory = Path(directory) if directory is not None else Path(__file__).resolve().parent\n        self.metadata = json.loads((self.directory / "models.json").read_text())\n        for name, expected in self.metadata["files"].items():\n            path = self.directory / name\n            if hashlib.sha256(path.read_bytes()).hexdigest() != expected["sha256"]:\n                raise ValueError(f"Model fingerprint mismatch: {name}")\n        self.models, self.addresses = {}, {}\n        for workload, rules in self.metadata["predictors"].items():\n            for rule, recipe in rules.items():\n                key = (workload, rule)\n                if recipe["method"] in ("log_l2", "raw_l1", "mix_log25", "mix_log50", "mix_log75"):\n                    self.models[key] = {\n                        method: lgb.Booster(model_file=str(self.directory / name))\n                        for method, name in recipe["models"].items()}\n                elif recipe["method"] == "address_median":\n                    saved = json.loads((self.directory / recipe["address_medians"]).read_text())\n                    self.addresses[key] = {int(address): value for address, value in saved.items()}\n                else:\n                    raise ValueError(f"Unsupported selected method: {recipe[\'method\']}")\n\n    def predict_seconds(self, workload, rule, raw_features):\n        values = np.atleast_2d(np.asarray(raw_features, dtype=np.float64))\n        if values.ndim != 2 or values.shape[1] != 7:\n            raise ValueError("Expected (n, 7) raw features in the order documented in models.json")\n        key = (workload, rule)\n        recipe = self.metadata["predictors"][workload][rule]\n        if recipe["method"] == "address_median":\n            addresses = self.addresses[key]\n            return np.asarray([addresses.get(int(offset), recipe["global_median_seconds"])\n                               for offset in values[:, 0]], dtype=float)\n        features = np.log1p(values)\n        models = self.models[key]\n        if recipe["method"] == "log_l2":\n            return np.maximum(0, np.expm1(models["log_l2"].predict(features, num_threads=2)))\n        if recipe["method"] == "raw_l1":\n            return np.maximum(0, models["raw_l1"].predict(features, num_threads=2))\n        log_seconds = np.maximum(0, np.expm1(models["log_l2"].predict(features, num_threads=2)))\n        raw_seconds = np.maximum(0, models["raw_l1"].predict(features, num_threads=2))\n        weight = recipe["weight_log_l2"]\n        return weight * log_seconds + (1 - weight) * raw_seconds\n'

In [ ]:
def require_grid(offsets, sizes):
    offsets = np.asarray(offsets, dtype=np.int64)
    sizes = np.asarray(sizes, dtype=np.int64)
    if (offsets < 0).any() or (sizes <= 0).any() or (offsets % BLOCK).any() or (sizes % BLOCK).any():
        raise ValueError("Events must have nonnegative 512-byte aligned offsets and positive aligned sizes")

In [ ]:
def _expand_blocks(starts, ends, indices):
    """Per-event half-open block spans to flat (block, index) pairs."""
    counts = np.maximum(ends - starts, 0)
    total = int(counts.sum())
    if not total:
        empty_i = np.empty(0, dtype=np.int64)
        return empty_i, empty_i
    block = np.repeat(starts - np.cumsum(counts, dtype=np.int64) + counts, counts) + np.arange(total, dtype=np.int64)
    pair_index = np.repeat(indices, counts)
    return block, pair_index

In [ ]:
def _keys(blocks, indices):
    return (blocks.astype(np.int64) << _INDEX_BITS) | (_INDEX_MOD - 1 - indices)

In [ ]:
def first_future_intersecting(query_starts, query_ends, query_indices,
                              candidate_starts, candidate_ends, candidate_indices,
                              regions=16):
    """For each query range, the smallest candidate row index greater than the
    query's own index whose byte range intersects the query range; -1 if none.

    Exact on the 512-byte grid: candidates are expanded to per-block pairs and
    processed region by region to bound memory. Key packing assumes more than
    2**31 events and fewer than 2**32 blocks (both hold for these traces).
    """
    query_starts = np.asarray(query_starts, dtype=np.int64)
    query_ends = np.asarray(query_ends, dtype=np.int64)
    query_indices = np.asarray(query_indices, dtype=np.int64)
    candidate_starts = np.asarray(candidate_starts, dtype=np.int64)
    candidate_ends = np.asarray(candidate_ends, dtype=np.int64)
    candidate_indices = np.asarray(candidate_indices, dtype=np.int64)
    if (query_indices >= _INDEX_MOD - 1).any() or (candidate_indices >= _INDEX_MOD - 1).any():
        raise ValueError("Row indices must fit the packed key width")
    result = np.full(len(query_indices), np.iinfo(np.int64).max, dtype=np.int64)
    if not len(candidate_indices) or not len(query_indices):
        return np.where(result == np.iinfo(np.int64).max, -1, result)
    q_begin, q_end = query_starts // BLOCK, query_ends // BLOCK
    c_begin, c_end = candidate_starts // BLOCK, candidate_ends // BLOCK
    span = max(int(c_end.max()), int(q_end.max()))
    c_total = int((c_end - c_begin).sum())
    needed = 1
    while needed * 30_000_000 < c_total:
        needed *= 2
    regions = max(regions, needed)
    region_size = -(-span // regions)
    for lo in range(0, span, region_size):
        hi = lo + region_size
        csel = np.flatnonzero((c_begin < hi) & (c_end > lo))
        qsel = np.flatnonzero((q_begin < hi) & (q_end > lo))
        if not len(csel) or not len(qsel):
            continue
        cblocks, cindex = _expand_blocks(
            np.maximum(c_begin[csel], lo), np.minimum(c_end[csel], hi), candidate_indices[csel])
        ckeys = _keys(cblocks, cindex)
        ckeys.sort()
        if int(ckeys.nbytes) > 6_400_000_000:
            raise ValueError("Region pair budget exceeded; increase regions")
        qblocks, qindex = _expand_blocks(
            np.maximum(q_begin[qsel], lo), np.minimum(q_end[qsel], hi), query_indices[qsel])
        qkeys = _keys(qblocks, qindex)
        position = np.searchsorted(ckeys, qkeys, side="left")
        previous = position - 1
        valid = (previous >= 0) & (ckeys[np.maximum(previous, 0)] >> _INDEX_BITS == qblocks)
        found = np.where(valid, _INDEX_MOD - 1 - (ckeys[np.maximum(previous, 0)] & _INDEX_MASK),
                         np.iinfo(np.int64).max)
        boundaries = np.zeros(len(qsel) + 1, dtype=np.int64)
        np.cumsum(np.minimum(q_end[qsel], hi) - np.maximum(q_begin[qsel], lo), out=boundaries[1:])
        per_query = np.minimum.reduceat(found, boundaries[:-1])
        result[qsel] = np.minimum(result[qsel], per_query)
    return np.where(result == np.iinfo(np.int64).max, -1, result)

In [ ]:
def first_future_same_start(query_offsets, query_indices, candidate_offsets, candidate_indices):
    """Previous-stage rule: first later candidate with the identical start offset."""
    query_offsets = np.asarray(query_offsets, dtype=np.int64)
    query_indices = np.asarray(query_indices, dtype=np.int64)
    candidate_offsets = np.asarray(candidate_offsets, dtype=np.int64)
    candidate_indices = np.asarray(candidate_indices, dtype=np.int64)
    result = np.full(len(query_offsets), -1, dtype=np.int64)
    if not len(candidate_indices) or not len(query_offsets):
        return result
    unique = np.unique(candidate_offsets)
    rank = np.searchsorted(unique, candidate_offsets)
    keys = (rank.astype(np.int64) << _INDEX_BITS) | candidate_indices
    order = np.argsort(keys, kind="stable")
    keys = keys[order]
    sorted_candidates = candidate_indices[order]
    qrank = np.searchsorted(unique, query_offsets)
    present = (qrank < len(unique)) & (unique[np.minimum(qrank, len(unique) - 1)] == query_offsets)
    qkeys = (qrank.astype(np.int64) << _INDEX_BITS) | (query_indices + 1)
    position = np.searchsorted(keys, qkeys, side="left")
    valid = present & (position < len(keys))
    safe = np.minimum(position, len(keys) - 1)
    same = valid & (keys[safe] >> _INDEX_BITS == qrank)
    result[same] = sorted_candidates[safe[same]]
    return result

In [ ]:
def label_writes(events, write_rows, begin, regions=16):
    """Range labels for query rows against every later event from `begin` on.

    Returns read/overwrite absolute row indices (-1 if none) and issue-to-issue
    target/lifetime in seconds (NaN when unmatched). Exact on the 512-byte grid.
    The event array is used as a read-only view; only needed columns are copied.
    """
    write_rows = np.asarray(write_rows, dtype=np.int64)
    if (write_rows < begin).any():
        raise ValueError("Query rows must not precede the candidate begin")
    offset = np.asarray(events["offset"][write_rows])
    size = np.asarray(events["size"][write_rows])
    timestamp = np.asarray(events["timestamp"][write_rows])
    is_read = np.asarray(events["is_read"][begin:]) == 1
    read_rows = np.flatnonzero(is_read) + begin
    write_candidate_rows = np.flatnonzero(~is_read) + begin
    read_off = np.asarray(events["offset"][read_rows])
    read_size = np.asarray(events["size"][read_rows])
    write_off = np.asarray(events["offset"][write_candidate_rows])
    write_size = np.asarray(events["size"][write_candidate_rows])
    read_idx = first_future_intersecting(offset, offset + size, write_rows, read_off, read_off + read_size, read_rows, regions=regions)
    write_idx = first_future_intersecting(offset, offset + size, write_rows, write_off, write_off + write_size, write_candidate_rows, regions=regions)

    def delay(match_rows):
        seconds = np.full(len(match_rows), np.nan)
        present = match_rows >= 0
        if present.any():
            seconds[present] = (np.asarray(events["timestamp"][match_rows[present]]) - timestamp[present]) / TICKS
        return seconds

    return {"read": read_idx, "overwrite": write_idx, "target": delay(read_idx), "lifetime": delay(write_idx)}

In [ ]:
def local_labels(events, begin, end, rule):
    rows = np.flatnonzero(events["is_read"][begin:end] == 0) + begin
    if rule == "range":
        return rows, label_writes(events[:end], rows, begin)
    reads = np.flatnonzero(events["is_read"][begin:end] == 1) + begin
    labels = {}
    for name, candidates in (("read", reads), ("overwrite", rows)):
        labels[name] = first_future_same_start(
            events["offset"][rows], rows, events["offset"][candidates], candidates)
    for index_name, delay_name in (("read", "target"), ("overwrite", "lifetime")):
        present = labels[index_name] >= 0
        delay = np.full(len(rows), np.nan)
        delay[present] = (events["timestamp"][labels[index_name][present]]
                          - events["timestamp"][rows[present]]) / TICKS
        labels[delay_name] = delay
    return rows, labels

In [ ]:
def prepare_samples(events, features_path, batch_size=250_000):
    address_order = np.argsort(events["offset"], kind="stable")
    sorted_read_flags = events["is_read"][address_order].astype(bool)
    read_positions = np.flatnonzero(sorted_read_flags)
    write_positions = np.flatnonzero(~sorted_read_flags)
    read_event_indices = address_order[read_positions]
    write_event_indices = address_order[write_positions]
    del address_order, sorted_read_flags

    read_offsets = events["offset"][read_event_indices]
    write_offsets = events["offset"][write_event_indices]
    chronological_writes = np.flatnonzero(events["is_read"] == 0)
    write_count = len(chronological_writes)

    features = np.lib.format.open_memmap(
        features_path,
        mode="w+",
        dtype=np.float64,
        shape=(write_count, len(FEATURE_NAMES)),
    )
    next_read_indices = np.full(write_count, -1, dtype=np.int64)
    target_seconds = np.full(write_count, np.nan)

    for batch_start in range(0, write_count, batch_size):
        batch_end = min(batch_start + batch_size, write_count)
        sorted_write_rows = np.arange(batch_start, batch_end)
        write_indices = write_event_indices[batch_start:batch_end]
        offsets = write_offsets[batch_start:batch_end]
        write_timestamps = events["timestamp"][write_indices]
        feature_rows = np.searchsorted(chronological_writes, write_indices)

        next_read_slots = np.searchsorted(
            read_positions,
            write_positions[batch_start:batch_end],
            side="right",
        )
        prior_read_counts = next_read_slots - np.searchsorted(read_offsets, offsets)
        prior_write_counts = sorted_write_rows - np.searchsorted(write_offsets, offsets)

        raw_features = np.full((batch_end - batch_start, len(FEATURE_NAMES)), np.nan)
        raw_features[:, 0] = offsets
        raw_features[:, 1] = events["size"][write_indices]
        raw_features[:, 2] = prior_read_counts
        raw_features[:, 3] = prior_write_counts

        previous_writes = write_event_indices[np.maximum(sorted_write_rows - 1, 0)]
        has_previous_write = prior_write_counts > 0
        raw_features[has_previous_write, 5] = (
            write_timestamps[has_previous_write]
            - events["timestamp"][previous_writes[has_previous_write]]
        ) / TICKS_PER_SECOND

        if len(read_event_indices):
            last_read_slot = len(read_event_indices) - 1
            previous_reads = read_event_indices[np.clip(next_read_slots - 1, 0, last_read_slot)]
            earlier_reads = read_event_indices[np.clip(next_read_slots - 2, 0, last_read_slot)]
            has_previous_read = prior_read_counts > 0
            has_previous_interval = prior_read_counts > 1

            raw_features[has_previous_read, 4] = (
                write_timestamps[has_previous_read]
                - events["timestamp"][previous_reads[has_previous_read]]
            ) / TICKS_PER_SECOND
            raw_features[has_previous_interval, 6] = (
                events["timestamp"][previous_reads[has_previous_interval]]
                - events["timestamp"][earlier_reads[has_previous_interval]]
            ) / TICKS_PER_SECOND

            candidate_reads = read_event_indices[np.minimum(next_read_slots, last_read_slot)]
            has_next_read = (
                (next_read_slots < len(read_event_indices))
                & (events["offset"][candidate_reads] == offsets)
            )
            completed_rows = feature_rows[has_next_read]
            next_read_indices[completed_rows] = candidate_reads[has_next_read]
            target_seconds[completed_rows] = (
                events["timestamp"][candidate_reads[has_next_read]]
                - write_timestamps[has_next_read]
            ) / TICKS_PER_SECOND

        features[feature_rows] = np.log1p(raw_features)

    features.flush()
    return {
        "features": features,
        "write_indices": chronological_writes,
        "next_read_indices": next_read_indices,
        "targets": target_seconds,
    }

In [ ]:
def fit_baselines(offsets, targets):
    """Frozen medians of completed training intervals; no evaluation truth."""
    median = float(np.median(targets))
    groups = {}
    for offset, target in zip(offsets, targets):
        groups.setdefault(int(offset), []).append(float(target))
    return median, {offset: float(np.median(values)) for offset, values in groups.items()}

In [ ]:
def load_four_days(dataset_directory, workload, cache_path, max_events=100_000_000):
    archive_name, member_name = TRACE_SOURCES[workload]
    buffer = array("q")
    if buffer.itemsize != 8 or not np.little_endian:
        raise RuntimeError("The event cache requires little-endian 64-bit integers")
    count, origin, previous, volume = 0, None, None, None
    reached_boundary = False
    with tarfile.open(Path(dataset_directory) / archive_name) as archive, cache_path.open("wb") as cache:
        member = archive.extractfile(member_name)
        if member is None:
            raise ValueError(f"Missing trace: {member_name}")
        with member, gzip.GzipFile(fileobj=member) as trace:
            for line in trace:
                fields = line.strip().split(b",")
                if len(fields) != 7:
                    raise ValueError(f"Invalid row: {count + 1}")
                timestamp, host, disk, operation, offset, size, response = fields
                timestamp, offset, size = int(timestamp), int(offset), int(size)
                current_volume = (host.decode("ascii"), int(disk))
                if origin is None:
                    origin, volume = timestamp, current_volume
                if previous is not None and timestamp < previous:
                    raise ValueError("Timestamps must be chronological")
                if current_volume != volume:
                    raise ValueError("Multiple logical volumes")
                if timestamp >= origin + 4 * 86400 * TICKS:
                    reached_boundary = True
                    break
                if count >= max_events:
                    raise ValueError("Event cap reached before four-day boundary")
                if operation not in (b"Read", b"Write") or offset < 0 or size <= 0 or int(response) < 0:
                    raise ValueError("Invalid I/O event")
                buffer.extend((timestamp, offset, size, int(operation == b"Read")))
                count += 1
                previous = timestamp
                if count % 250_000 == 0:
                    buffer.tofile(cache)
                    buffer = array("q")
                if count % 2_000_000 == 0:
                    print(workload, f"{count:,} events", flush=True)
            buffer.tofile(cache)
    if not reached_boundary or not count:
        raise ValueError("Trace does not establish the full four-day observation window")
    return np.memmap(cache_path, mode="r", dtype=EVENT_DTYPE, shape=(count,))

In [ ]:
def validation_metrics(events, rows, labels, prediction, boundary):
    seen = labels["read"] >= 0
    if not seen.any():
        raise ValueError("No observed validation Read targets")
    known = labels["overwrite"][seen] >= 0
    error = np.abs(labels["target"][seen] - prediction[seen])
    remaining = (boundary - events["timestamp"][rows[seen]]) / TICKS
    lower_intervals = np.where(known, labels["lifetime"][seen], remaining)
    passed = error < lower_intervals
    uncertain = ~known & ~passed
    mae, lower = float(error.mean()), float(lower_intervals.mean())
    if mae < lower:
        mean_status = "passed"
    elif known.all():
        mean_status = "failed"
    else:
        mean_status = "undetermined"
    return {"n": int(seen.sum()), "all_writes": len(rows), "mae_seconds": mae,
            "mean_overwrite_lower_seconds": lower, "mean_status": mean_status,
            "pass_lower": float(passed.mean()), "pass_upper": float((passed | uncertain).mean())}

In [ ]:
def train_events(events, workload, directory, parameters, weights):
    require_grid(events["offset"], events["size"])
    origin = int(events["timestamp"][0])
    train_end, valid_end = map(int, np.searchsorted(events["timestamp"],
                                                   [origin + 3 * 86400 * TICKS, origin + 4 * 86400 * TICKS]))
    if not 0 < train_end < valid_end:
        raise ValueError("Train days 1–3 and validation day 4 must be nonempty")
    with TemporaryDirectory() as temporary:
        samples = prepare_samples(events[:valid_end], Path(temporary) / "features.npy")
        writes, features = samples["write_indices"], samples["features"]
        results = {}
        for rule in ("same_start", "range"):
            train_rows, train = local_labels(events, 0, train_end, rule)
            rows, valid = local_labels(events, train_end, valid_end, rule)
            seen = train["read"] >= 0
            if not seen.any():
                raise ValueError(f"No training Read targets: {workload}/{rule}")
            train_x = features[np.searchsorted(writes, train_rows[seen])]
            validation_x = features[np.searchsorted(writes, rows)]
            predictions, model_files = {}, {}
            for method in ("log_l2", "raw_l1"):
                objective = "regression" if method == "log_l2" else "regression_l1"
                target = np.log1p(train["target"][seen]) if method == "log_l2" else train["target"][seen]
                model = lgb.LGBMRegressor(objective=objective, **parameters)
                model.fit(train_x, target)
                name = f"{workload}.{rule}.{method}.lightgbm.txt"
                model.booster_.save_model(str(directory / name))
                model_files[method] = name
                values = model.booster_.predict(validation_x, num_threads=2)
                predictions[method] = np.maximum(0, np.expm1(values) if method == "log_l2" else values)
            median, addresses = fit_baselines(events["offset"][train_rows[seen]], train["target"][seen])
            address_file = f"{workload}.{rule}.address_medians.json"
            (directory / address_file).write_text(json.dumps({str(key): value for key, value in addresses.items()}, indent=2) + "\n")
            predictions["address_median"] = np.asarray([addresses.get(int(offset), median) for offset in events["offset"][rows]])
            if workload == "prn_1":
                predictions.update({name: weight * predictions["log_l2"] + (1 - weight) * predictions["raw_l1"]
                                    for name, weight in weights.items()})
            metrics = {name: validation_metrics(events, rows, valid, prediction, origin + 4 * 86400 * TICKS)
                       for name, prediction in predictions.items()}
            selected = min(metrics, key=lambda name: (metrics[name]["mae_seconds"], name))
            results[rule] = {"selected": selected, "training_samples": int(seen.sum()),
                             "validation": metrics, "models": model_files,
                             "address_medians": address_file, "global_median_seconds": median}
        del samples, features
    return results

In [ ]:
def train_latest(workload, dataset_directory, output_directory, parameters, weights):
    with TemporaryDirectory() as temporary:
        events = load_four_days(dataset_directory, workload, Path(temporary) / "events.bin")
        result = train_events(events, workload, output_directory, parameters, weights)
        del events
    return result

In [ ]:
def export_predictors(training_results, directory, weights, inference_source, training_mode):
    metadata = {"raw_feature_order": RAW_FEATURES, "input_transform": "log1p for LightGBM; raw offset for address median",
                "missing_history": "NaN", "output": "seconds from Write to first subsequent Read",
                "history_key": "same starting Offset for both labeling rules; before current Write",
                "train_window": "days 1–3", "selection_window": "day 4 observed-Read MAE",
                "training_mode": training_mode, "evaluation_role": "validation only; no future-day evaluation",
                "predictors": {}, "files": {}}
    for workload, rules in training_results.items():
        metadata["predictors"][workload] = {}
        for rule, result in rules.items():
            method = result["selected"]
            recipe = {"method": method}
            if method == "address_median":
                recipe.update(address_medians=result["address_medians"], global_median_seconds=result["global_median_seconds"])
                names = [result["address_medians"]]
            elif method in weights:
                recipe.update(models=result["models"], weight_log_l2=weights[method])
                names = list(result["models"].values())
            else:
                recipe["models"] = {method: result["models"][method]}
                names = list(recipe["models"].values())
            metadata["predictors"][workload][rule] = recipe
            for name in names:
                path = directory / name
                metadata["files"][name] = {"size_bytes": path.stat().st_size,
                                           "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    (directory / "predict.py").write_text(inference_source)
    path = directory / "predict.py"
    metadata["files"]["predict.py"] = {"size_bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    (directory / "models.json").write_text(json.dumps(metadata, indent=2) + "\n")
    (directory / "training-results.json").write_text(json.dumps(training_results, indent=2) + "\n")
    return metadata

In [ ]:
def smoke_events():
    """Small artificial four-day trace for checking the notebook, not evidence."""
    rows = []
    for day in range(4):
        for index in range(64):
            timestamp = (day * 86400 + index * 1200) * TICKS
            offset = (index % 8) * 512
            read_offset = offset + 512 if index % 3 == 0 else offset
            rows.extend([(timestamp, offset, 1024, 0),
                         (timestamp + (10 + index % 5) * TICKS, read_offset, 512, 1),
                         (timestamp + 30 * TICKS, offset, 512, 0)])
    return np.asarray(rows, dtype=EVENT_DTYPE)

## Training and export

Choose a new OUTPUT directory: an existing bundle is not overwritten. Outputs include LightGBM models, address medians, training-results.json with validation metrics, models.json with the selected method and fingerprints, and predict.py for inference. For unknown overwrite times, remaining observation time is used only as a lower bound. SMOKE_TEST checks execution on artificial data and is not evidence of predictive quality.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(exist_ok=False)
training_results = {}
for workload in WORKLOADS:
    if SMOKE_TEST:
        training_results[workload] = train_events(smoke_events(), workload, OUTPUT, PARAMETERS, MIX_WEIGHTS)
    else:
        training_results[workload] = train_latest(workload, DATASET, OUTPUT, PARAMETERS, MIX_WEIGHTS)
mode = 'artificial_smoke' if SMOKE_TEST else 'real_four_day'
exported = export_predictors(training_results, OUTPUT, MIX_WEIGHTS, INFERENCE_SOURCE, mode)
print('Mode:', 'ARTIFICIAL SMOKE TEST' if SMOKE_TEST else 'REAL FOUR-DAY TRAINING')
print('Exported predictors:', json.dumps(exported['predictors'], indent=2))

In [ ]:
from IPython.display import Markdown, display

rows = ['| Workload / rule | Candidate | Train N | Validation N | MAE, s | Mean overwrite ≥, s | Pass lower–upper | Selected |',
        '|---|---|---:|---:|---:|---:|---:|---|']
for workload, rules in training_results.items():
    for rule, result in rules.items():
        for method, metric in result['validation'].items():
            rows.append(f"| {workload} / {rule} | {method} | {result['training_samples']} | {metric['n']} | {metric['mae_seconds']:.2f} | {metric['mean_overwrite_lower_seconds']:.2f} | {metric['pass_lower']:.2%}–{metric['pass_upper']:.2%} | {method == result['selected']} |")
display(Markdown('\n'.join(rows)))

In [ ]:
spec = importlib.util.spec_from_file_location('trained_predictor', OUTPUT / 'predict.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
predictor = module.SelectedPredictor(OUTPUT)
raw_features = [[4096, 512, 0, 0, np.nan, np.nan, np.nan]]
for workload in WORKLOADS:
    for rule in ('same_start', 'range'):
        seconds = predictor.predict_seconds(workload, rule, raw_features)
        assert np.isfinite(seconds).all() and (seconds >= 0).all()
        print(workload, rule, 'illustrative prediction, seconds:', seconds)